
# Gradient Flow y conexiones residuales

En este notebook vamos a estudiar una sola idea:

> **¿Qué le ocurre al gradiente cuando una red se hace profunda y por qué las conexiones residuales pueden ayudar?**

No usaremos CIFAR-10 ni una tarea de clasificación.

Queremos aislar el mecanismo.

---

## Objetivos

Al terminar deberías poder explicar:

1. Qué significa **gradient flow**.
2. Qué es **vanishing gradient**.
3. Por qué una red profunda puede dificultar la propagación del gradiente.
4. Qué cambia cuando usamos:

\[
y = F(x) + x
\]

5. De dónde aparece el término:

\[
+1
\]

en la derivada de un bloque residual.
6. Por qué los gradientes de múltiples caminos se **suman**.
7. Cómo observar experimentalmente la magnitud del gradiente en distintas profundidades.


In [ ]:

import random

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)



# 1. ¿Qué significa gradient flow?

Durante el forward:

```text
x
↓
Layer 1
↓
Layer 2
↓
Layer 3
↓
...
↓
Loss
```

Durante backpropagation, la información viaja en sentido contrario:

```text
Loss
↑
Layer n
↑
...
↑
Layer 2
↑
Layer 1
↑
x
```

El **gradient flow** describe cómo se propaga esa señal de gradiente a través de la red.



# 2. Regla de la cadena y profundidad

Supongamos:

\[
x
\rightarrow h_1
\rightarrow h_2
\rightarrow \cdots
\rightarrow h_n
\rightarrow \mathcal L
\]

Entonces:

\[
\frac{\partial \mathcal L}{\partial x}
=
\frac{\partial \mathcal L}{\partial h_n}
\frac{\partial h_n}{\partial h_{n-1}}
\cdots
\frac{\partial h_2}{\partial h_1}
\frac{\partial h_1}{\partial x}
\]

Aparece un producto de muchas derivadas.

Si muchas tienen magnitud menor que 1:

```text
0.5 × 0.5 × 0.5 × 0.5 × ...
```

el resultado puede volverse muy pequeño.

Por ejemplo:

\[
0.5^{20}
\approx 9.5\times10^{-7}
\]

Este fenómeno se relaciona con:

\[
\boxed{\text{vanishing gradients}}
\]


In [ ]:

for n in [1, 2, 4, 8, 12, 20]:
    print(
        f"0.5^{n:2d} =",
        f"{0.5**n:.10f}"
    )



# 3. Una red plain

Una red profunda convencional puede verse como:

```text
x
↓
Block 1
↓
Block 2
↓
Block 3
↓
...
↓
Block n
```

Cada bloque transforma la representación que recibe.

Vamos a construir un bloque sencillo:

```text
Linear
↓
ReLU
↓
Linear
↓
ReLU
```


In [ ]:

class PlainBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.block(x)



# 4. La idea residual

ResNet introduce una ruta adicional:

\[
y = F(x) + x
\]

Visualmente:

```text
x ───────────────────────┐
│                        │
│       F(x)             │
└── capas aprendibles ─ (+)
                         │
                         ↓
                         y
```

En vez de depender únicamente de:

```text
x → F(x) → y
```

tenemos también:

```text
x ─────────────→ y
```

Ese camino directo es el **shortcut**.



# 5. ¿Qué ocurre durante backpropagation?

Si:

\[
y = F(x) + x
\]

entonces:

\[
\frac{\partial y}{\partial x}
=
\frac{\partial F(x)}{\partial x}
+
\frac{\partial x}{\partial x}
\]

pero:

\[
\frac{\partial x}{\partial x}=1
\]

por tanto:

\[
\boxed{
\frac{\partial y}{\partial x}
=
\frac{\partial F(x)}{\partial x}+1
}
\]

Y aplicando la regla de la cadena:

\[
\boxed{
\frac{\partial \mathcal L}{\partial x}
=
\frac{\partial \mathcal L}{\partial y}
\left(
\frac{\partial F(x)}{\partial x}+1
\right)
}
\]

Ese término:

\[
\boxed{+1}
\]

proviene directamente del shortcut identidad.



# 6. Los gradientes de varios caminos se suman

La entrada \(x\) influye en \(y\) mediante dos caminos:

```text
                    camino F(x)
                 ┌──────────────┐
                 │              ▼
x ───────────────┼──────────── (+) → y → Loss
│                │              ▲
│                └──────────────┘
│
└──────────── shortcut ──────────┘
```

Durante backpropagation:

```text
                      ∂L/∂y
                         │
              ┌──────────┴──────────┐
              │                     │
              ▼                     ▼
         camino F(x)         shortcut identidad
              │                     │
              ▼                     ▼
           ∂F/∂x                    1
              │                     │
              └──────────┬──────────┘
                         ▼
                      SE SUMAN
                         │
                         ▼
                       ∂L/∂x
```

Esta es una regla general:

> Cuando una variable contribuye a la loss por varios caminos, el gradiente total es la suma de las contribuciones de esos caminos.



# 7. Implementemos un bloque residual

Usaremos:

```text
x ─────────────────────┐
│                      │
Linear → ReLU → Linear │
│                      │
└──────────────────── (+)
                       ↓
                     ReLU
```


In [ ]:

class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        identity = x

        out = self.fc1(x)
        out = self.relu(out)

        out = self.fc2(out)

        out = out + identity
        out = self.relu(out)

        return out



# 8. Red plain vs red residual

Construiremos dos redes con:

- la misma dimensión interna;
- el mismo número de bloques;
- una profundidad comparable.

### Plain

```text
Block
↓
Block
↓
Block
↓
...
```

### Residual

```text
ResidualBlock
↓
ResidualBlock
↓
ResidualBlock
↓
...
```


In [ ]:

class DeepPlainMLP(nn.Module):
    def __init__(
        self,
        dim=64,
        n_blocks=12,
    ):
        super().__init__()

        self.blocks = nn.ModuleList([
            PlainBlock(dim)
            for _ in range(n_blocks)
        ])

    def forward(
        self,
        x,
        keep_activations=False,
    ):
        activations = []

        for block in self.blocks:
            x = block(x)

            if keep_activations:
                x.retain_grad()
                activations.append(x)

        return x, activations


class DeepResidualMLP(nn.Module):
    def __init__(
        self,
        dim=64,
        n_blocks=12,
    ):
        super().__init__()

        self.blocks = nn.ModuleList([
            ResidualBlock(dim)
            for _ in range(n_blocks)
        ])

    def forward(
        self,
        x,
        keep_activations=False,
    ):
        activations = []

        for block in self.blocks:
            x = block(x)

            if keep_activations:
                x.retain_grad()
                activations.append(x)

        return x, activations



# 9. ¿Por qué usamos `retain_grad()`?

PyTorch calcula gradientes para todas las operaciones necesarias durante backpropagation.

Pero por defecto conserva `.grad` principalmente para tensores hoja.

Las activaciones intermedias:

```text
h1
h2
h3
...
```

no guardan automáticamente su gradiente.

Para inspeccionarlo usamos:

```python
activation.retain_grad()
```

Esto **no cambia backpropagation**.

Solo nos permite observar el gradiente después de:

```python
loss.backward()
```



# 10. Experimento controlado

No queremos resolver una tarea real.

Vamos a usar:

- dimensión: `64`;
- bloques: `12`;
- batch: `32`;
- target aleatorio;
- MSE como loss;
- un solo forward/backward.

La pregunta es únicamente:

> ¿Cuánto gradiente llega a cada profundidad?


In [ ]:

torch.manual_seed(SEED)

DIM = 64
N_BLOCKS = 12
BATCH_SIZE = 32

x_plain = torch.randn(
    BATCH_SIZE,
    DIM,
    requires_grad=True,
)

x_residual = (
    x_plain
    .detach()
    .clone()
    .requires_grad_(True)
)

target = torch.randn(
    BATCH_SIZE,
    DIM,
)

plain_model = DeepPlainMLP(
    dim=DIM,
    n_blocks=N_BLOCKS,
)

residual_model = DeepResidualMLP(
    dim=DIM,
    n_blocks=N_BLOCKS,
)

criterion = nn.MSELoss()



# 11. Forward y backward — red plain


In [ ]:

plain_output, plain_activations = plain_model(
    x_plain,
    keep_activations=True,
)

plain_loss = criterion(
    plain_output,
    target,
)

plain_loss.backward()

print(
    "Plain loss:",
    plain_loss.item(),
)



# 12. Forward y backward — red residual


In [ ]:

residual_output, residual_activations = residual_model(
    x_residual,
    keep_activations=True,
)

residual_loss = criterion(
    residual_output,
    target,
)

residual_loss.backward()

print(
    "Residual loss:",
    residual_loss.item(),
)



# 13. Norma del gradiente por profundidad

Para cada activación intermedia \(h_l\):

\[
\left\|
\nabla_{h_l}\mathcal L
\right\|_2
\]

nos da una medida de la magnitud del gradiente que llega hasta ese punto.

El bloque `1` está cerca de la entrada.

El bloque `12` está cerca de la loss.


In [ ]:

plain_grad_norms = [
    activation.grad.norm().item()
    for activation in plain_activations
]

residual_grad_norms = [
    activation.grad.norm().item()
    for activation in residual_activations
]

depth = np.arange(
    1,
    N_BLOCKS + 1,
)

plt.figure(figsize=(9, 5))

plt.plot(
    depth,
    plain_grad_norms,
    marker="o",
    label="Plain",
)

plt.plot(
    depth,
    residual_grad_norms,
    marker="o",
    label="Residual",
)

plt.yscale("log")

plt.xlabel("Salida del bloque")
plt.ylabel(
    r"$||\nabla_{h_l}\mathcal{L}||_2$"
)

plt.title(
    "Gradient flow: Plain vs Residual"
)

plt.legend()
plt.grid(False)
plt.show()



## ¿Cómo leer esta gráfica?

Forward:

```text
entrada → block 1 → block 2 → ... → block 12 → loss
```

Backward:

```text
loss → block 12 → ... → block 2 → block 1 → entrada
```

Por eso interesa especialmente observar qué ocurre cerca del:

```text
block 1
```

Si el gradiente allí es extremadamente pequeño, las primeras capas reciben poca señal para actualizar sus parámetros.



# 14. Gradiente que llega hasta la entrada

También podemos observar directamente:

\[
\left\|
\frac{\partial\mathcal L}{\partial x}
\right\|_2
\]

Esto resume cuánto gradiente logró regresar a través de toda la red.


In [ ]:

plain_input_grad = (
    x_plain.grad.norm().item()
)

residual_input_grad = (
    x_residual.grad.norm().item()
)

print(
    "Plain input gradient:",
    f"{plain_input_grad:.10f}"
)

print(
    "Residual input gradient:",
    f"{residual_input_grad:.10f}"
)

if plain_input_grad > 0:
    print(
        "\nResidual / Plain:",
        f"{residual_input_grad / plain_input_grad:.2f}x"
    )



# 15. ¿Qué podemos concluir?

La idea importante **no** es:

```text
ResNet siempre produce gradientes grandes.
```

Tampoco:

```text
ResNet elimina completamente vanishing gradients.
```

La conclusión más precisa es:

> **Las conexiones residuales proporcionan caminos adicionales por los que pueden propagarse señales y gradientes.**

Eso hace mucho más fácil optimizar arquitecturas profundas.

La arquitectura modifica la geometría del problema de optimización.



# 16. Experimento: aumenta la profundidad

Ahora repite el experimento con:

```python
N_BLOCKS = 2
N_BLOCKS = 4
N_BLOCKS = 8
N_BLOCKS = 16
N_BLOCKS = 32
```

y registra:

\[
\left\|
\frac{\partial\mathcal L}{\partial x}
\right\|_2
\]

para ambas redes.

### Hipótesis

Antes de ejecutar:

> ¿Qué esperas que ocurra con el gradiente de la red plain al aumentar la profundidad?

> ¿Y con la residual?


In [ ]:

# TODO
#
# depths = [2, 4, 8, 16, 32]
#
# plain_input_gradients = []
# residual_input_gradients = []
#
# for n_blocks in depths:
#     ...



# 17. Experimento opcional: sin ReLU

Modifica ambos bloques para eliminar temporalmente `ReLU`.

### Pregunta

¿El comportamiento del gradiente cambia?

Esto ayuda a separar dos factores:

```text
profundidad
+
activaciones no lineales
```

de:

```text
shortcut residual
```



# 18. Preguntas para discusión

1. ¿Qué significa **gradient flow**?
2. ¿Por qué la profundidad puede dificultar backpropagation?
3. ¿Qué relación existe entre vanishing gradients y productos de derivadas?
4. ¿De dónde aparece el `+1` en un bloque residual?
5. ¿Qué representa el shortcut en el forward?
6. ¿Qué representa en el backward?
7. ¿Por qué los gradientes de dos caminos se suman?
8. ¿Qué significa una norma de gradiente cercana a cero?
9. ¿Una conexión residual agrega necesariamente parámetros?
10. ¿Una ResNet garantiza que nunca habrá vanishing gradients?
11. ¿Por qué este mecanismo permite entrenar redes mucho más profundas?



# Mapa conceptual

```text
RED PROFUNDA
    │
    ▼
muchas reglas de la cadena
    │
    ▼
producto de derivadas
    │
    ▼
gradiente puede disminuir
    │
    ▼
capas tempranas reciben
poca señal de aprendizaje


RESIDUAL BLOCK
    │
    ├──────────────┐
    ▼              │
   F(x)            x
    │              │
    └──────┬───────┘
           ▼
         F(x)+x
           │
           ▼
      dos caminos
     para el gradiente
           │
           ▼
  gradientes se suman
           │
           ▼
facilita entrenamiento profundo
```
